# Phase 2 - Positions study

This notebook checks whether the relative positions of the dependency graph (level 3) can be measured from the labels of Phase 1.

## Section 1 - Purpose and questions

The dependency graph describes the correct assembly in levels. Level 1 lists the parts, level 2 says on which part each one is mounted, level 3 adds optional relative positions: above, right of, left of.

Level 3 is needed to see placement errors, such as the nose in the place of the eyes. Without it, a nose mounted on the body looks correct wherever it is.

The positions are not computed by a model. C-RADIO only labels each patch. The relation is computed from the centre of each part region on the 64 x 64 grid: "A above B" means that the centre of A is higher in the image than the centre of B.

The video comes from the headset (Meta Quest 3), so the camera moves with the head. The method may not hold: a tilted head, a view from above, wrong regions and unknown front and back can all turn a correct relation into a violated one.

This study measures the method on assembly_2, the development video. The assembly in assembly_2 is always correct, so every violated relation is a false alarm.

Three questions are asked. How often is a correct relation found violated? On how many frames can each relation be checked? How far apart are the two centres, in patches?

The vertical relations are studied first, because they do not need to know where the front of the figure is. The relations on the sides come second.

This is an exploration, not the decision. No threshold is fixed here. If false alarms are frequent, level 3 is removed and the framework works at level 2. If they are rare, level 3 is kept and its rule is fixed later with the procedure of the evaluation plan.

------------------------------------------------------------------------------------------------------------------------------------------

References Section 1:

Jones, Hager, Khudanpur, Toward Computer Vision Systems That Understand Real-World Assembly Processes, WACV 2019: the assembly state as a graph of parts and connections, with the relative pose on each connection, evaluated at levels of spatial detail.

Kamath, Hessel, Chang, What's "up" with vision-language models? Investigating their struggle with spatial reasoning, EMNLP 2023: vision-language models fail on simple spatial relations, so the relations are computed by code.

configs/dependency_graph.json: the positions, marked "to be validated".

------------------------------------------------------------------------------------------------------------------------------------------

## Section 2 - Configuration

All paths and settings in one place. The settings are those of the pipeline (phase_1_perception.ipynb, Section 2.5), so that A is applied exactly as in Phase 1.

A is not rebuilt here. It is loaded from the prototypes saved by the pipeline (patch_prototypes.npz), as in the study of the support set by condensation.

The labels of assembly_2 come from the support-set study on V2 (key "current", which is A). Section 3 checks them on a few frames with the final code before they are used.

In [1]:
# 2.1 Imports, paths and shared module
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import cv2
from PIL import Image
from IPython.display import display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))   # makes src/perception importable from notebooks/

from perception.classes import CLASS_NAMES
from perception.features import FeatureExtractor
from perception.prototypes import load_patch_prototypes, class_similarities_gpu, assign_labels

MODEL_DIR = PROJECT_ROOT / "models" / "c_radio_weights"
VIDEO_PATH = PROJECT_ROOT / "data" / "raw_videos" / "assembly_2.mp4"
EVENTS_PATH = PROJECT_ROOT / "configs" / "assembly_2_events.json"
GRAPH_PATH = PROJECT_ROOT / "configs" / "dependency_graph.json"
PROTOTYPES_PATH = PROJECT_ROOT / "outputs" / "phase_1_perception" / "patch_prototypes.npz"   # support set A, read only
LABELS_PATH = PROJECT_ROOT / "outputs" / "phase_1_support_set_v2_study" / "assembly_2_labels_current_protected.npz"  # read only
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase_2_positions_study"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"MODEL_DIR": MODEL_DIR, "VIDEO_PATH": VIDEO_PATH, "EVENTS_PATH": EVENTS_PATH,
                   "GRAPH_PATH": GRAPH_PATH, "PROTOTYPES_PATH": PROTOTYPES_PATH, "LABELS_PATH": LABELS_PATH}.items():
    print(f"[{'OK' if path.exists() else 'MISSING'}] {name}: {path.relative_to(PROJECT_ROOT)}")

[OK] MODEL_DIR: models\c_radio_weights
[OK] VIDEO_PATH: data\raw_videos\assembly_2.mp4
[OK] EVENTS_PATH: configs\assembly_2_events.json
[OK] GRAPH_PATH: configs\dependency_graph.json
[OK] PROTOTYPES_PATH: outputs\phase_1_perception\patch_prototypes.npz
[OK] LABELS_PATH: outputs\phase_1_support_set_v2_study\assembly_2_labels_current_protected.npz


In [2]:
# 2.2 Settings
# Values of the pipeline (phase_1_perception.ipynb, Section 2.5); the input size is the one the extractor gives for these videos.
DEVICE, DTYPE = "cuda", torch.bfloat16
INPUT_SIZE, PATCH_SIZE = (1024, 1024), 16
MARGIN = 0.0                    # background margin of V2 (margin study, Section 9)
N_CHECK_FRAMES = 10             # frames of assembly_2 recomputed with the final code (Section 3)

Observation 2: The six paths exist: the C-RADIO weights, the video and the event log of assembly_2, the dependency graph, the prototypes of A and the saved labels of assembly_2. The settings are those of the pipeline: bfloat16 on the GPU, input 1024 x 1024, patches of 16 pixels, margin 0.

------------------------------------------------------------------------------------------------------------------------------------------

References Section 2:

phase_1_perception.ipynb, Section 2.5: the settings of the pipeline; Section 6.4: the prototypes of A saved in patch_prototypes.npz.

phase_1_support_set_v2_study.ipynb, Section 10.1: the labels of assembly_2 with the support sets "current" (A) and "protected".

phase_1_support_set_hart.ipynb, Section 2.1: A loaded from the saved prototypes, read only.

------------------------------------------------------------------------------------------------------------------------------------------

## Section 3 - The labels of assembly_2

Phase 2 works on the labels of every frame, not on the video. For assembly_2 they were saved by the support-set study on V2 (version v1.7), with the support set "current", which is A: same 39,125 prototypes, margin 0, mIoU 0.779 in validation.

The code changed between v1.7 and the final pipeline. The labels are therefore checked before use: A is applied again, with the final code, to a few frames of assembly_2, and the result is compared patch by patch with the saved labels.

The frames are fixed by a rule, not chosen by looking: 10 frames evenly spaced over the whole video. If no patch differs, the saved labels are used. If any patch differs, the labels of the whole video are computed again.

In [3]:
# 3.1 Support set A (saved prototypes of the pipeline) and the saved labels of assembly_2
from perception.labels import sequence_of

saved = load_patch_prototypes(PROTOTYPES_PATH)
A_VECTORS, A_OWNERS = saved["prototypes"], saved["owners"]
reference_sequences = {sequence_of(r["frame"]) for r in saved["metadata"]["reference_frames"]}
assert saved["class_names"] == CLASS_NAMES, "Class order in the file must match CLASS_NAMES"
assert not reference_sequences & {"assembly_1", "assembly_2"}, "A must not use the videos of Phase 2"
A_GPU = tuple(torch.as_tensor(x, device=DEVICE) for x in (A_VECTORS, A_OWNERS))
print(f"A: {len(A_VECTORS)} prototypes from {len(reference_sequences)} reference sequences: {sorted(reference_sequences)}")

stored = np.load(LABELS_PATH)
LABELS = stored["current"]
assert [str(c) for c in stored["class_names"]] == CLASS_NAMES, "Class order of the saved labels must match CLASS_NAMES"

cap = cv2.VideoCapture(str(VIDEO_PATH))
FPS, N_FRAMES = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
GRID = (INPUT_SIZE[0] // PATCH_SIZE, INPUT_SIZE[1] // PATCH_SIZE)
assert LABELS.shape == (N_FRAMES, *GRID), f"Saved labels {LABELS.shape} do not match the video"
print(f"Saved labels: {LABELS.shape[0]} frames, grid {GRID}; video: {N_FRAMES} frames at {FPS:.3f} fps "
      f"(saved fps {float(stored['fps']):.3f})")

A: 39125 prototypes from 6 reference sequences: ['2_disassembled_caos', '3_disassembled_ordered', '4_partially_assembled', '5_assembled', 'Take_3_1_Base', 'caos']
Saved labels: 2444 frames, grid (64, 64); video: 2444 frames at 29.996 fps (saved fps 29.996)


In [ ]:
# 3.2 A applied again, with the final code, to 10 frames evenly spaced over assembly_2
CHECK_FRAMES = np.linspace(0, N_FRAMES - 1, N_CHECK_FRAMES).round().astype(int)

extractor = FeatureExtractor(MODEL_DIR, device=DEVICE, dtype=DTYPE)
assert extractor.patch_size == PATCH_SIZE

recomputed = {}
start = time.time()
cap = cv2.VideoCapture(str(VIDEO_PATH))
for f in range(N_FRAMES):                       # frames are read in order: seeking can land on the wrong frame
    ok, frame_bgr = cap.read()
    if not ok:
        break
    if f in CHECK_FRAMES:
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        features, size = extractor.extract(Image.fromarray(frame_rgb))
        assert tuple(size) == INPUT_SIZE
        recomputed[f] = assign_labels(class_similarities_gpu(features, *A_GPU), MARGIN)
    if f >= CHECK_FRAMES[-1]:
        break
cap.release()
print(f"Frames checked: {CHECK_FRAMES.tolist()}")
print(f"{len(recomputed)} frames recomputed in {time.time() - start:.0f} s")

Loading weights:   0%|          | 0/330 [00:00<?, ?it/s]

Frames checked: [np.int64(0), np.int64(271), np.int64(543), np.int64(814), np.int64(1086), np.int64(1357), np.int64(1629), np.int64(1900), np.int64(2172), np.int64(2443)]
10 frames recomputed in 12 s
